# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```
The Croissant schema contains detailed metadata and provides access to multiple record sets, fields, and columns. All entities are referenced by their `@id` as per schema best practices.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata (as an object, not a dict)
meta = dataset.metadata
print(f"Dataset: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}")

## 2. Data Overview
List all available `RecordSet`s and their `@id`s, then explore their fields. All entities are referenced by their `@id` fields as per best practice.

In [ ]:
# List all available record sets with their @id and name
record_set_list = list(dataset.record_sets)
print("Available record sets and their @id:")
for rs in record_set_list:
    print(f"- @id: {rs['@id']} | name: {getattr(rs, 'name', 'N/A')}")

# For demonstration, select the first (and likely primary) record set.

primary_record_set_id = record_set_list[0]['@id'] if record_set_list else None
print(f"\nPrimary record set selected: {primary_record_set_id}")

# List the fields of this primary record set by @id and name
if primary_record_set_id:
    record_set = dataset.get_record_set(primary_record_set_id)
    field_ids = [field['@id'] for field in record_set.fields]
    print(f"\nFields in record set {primary_record_set_id}:")
    for field in record_set.fields:
        # Try to get informative name/label if exists
        print(f"  - @id: {field['@id']} | name: {field.get('name', field.get('rdfs:label', 'N/A'))}")

## 3. Data Extraction
Load data from each available RecordSet by their `@id`. Store results as Pandas DataFrames, using the record set `@id` as DataFrame keys.

In [ ]:
# Extract data from each record set identified in the overview
dataframes = {}

for rs in record_set_list:
    record_set_id = rs['@id']
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        if len(records) > 0:
            print(f"  Loaded {len(records)} records for {record_set_id}")
    except Exception as e:
        print(f"  Failed to load records for {record_set_id}: {e}")

if primary_record_set_id in dataframes:
    print(f"\nColumns (field @id) in primary record set {primary_record_set_id}:")
    print(list(dataframes[primary_record_set_id].columns))
    display(dataframes[primary_record_set_id].head())  # Display first 5 records

## 4. Exploratory Data Analysis (EDA)
Apply some typical processing/analysis using field `@id`s. We'll select a notable numeric field and a group/categorical field from the DataFrame columns, filter, normalize, and group as appropriate.

In [ ]:
import numpy as np

# List field @ids for numeric/categorical selection
columns = list(dataframes[primary_record_set_id].columns)
print(f"Data columns (field @id) in primary record set:\n{columns}\n")

# -- Update these with field @id (not label): e.g. 'age', 'tumor_stage', etc. --
# We'll try some common possibilities and fall back if needed.

# Choose numeric field (example: '@id' containing 'Age' or similar)
numeric_field_id = None
for col in columns:
    lower = col.lower()
    if 'age' in lower:
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try another numeric type (e.g. interval, count)
    for col in columns:
        if 'interval' in col.lower() or 'count' in col.lower() or 'number' in col.lower():
            numeric_field_id = col
            break
if numeric_field_id is None and columns:
    # Fallback: select the first numeric-looking column by inspecting values
    for col in columns:
        try:
            vals = pd.to_numeric(dataframes[primary_record_set_id][col], errors='coerce')
            if vals.notna().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

if numeric_field_id is None:
    raise ValueError("Could not determine a numeric field column from the data.")
print(f"Selected numeric field: {numeric_field_id}")

# Choose group/categorical field (default to one containing 'sex', 'msi', or similar as demonstration)
group_field_id = None
for col in columns:
    if any(x in col.lower() for x in ['sex', 'msi', 'site', 'location', 'group', 'anatomical']):
        group_field_id = col
        break
if group_field_id is None:
    # Fallback: choose any field with low unique-count
    for col in columns:
        if dataframes[primary_record_set_id][col].nunique() <= 5:
            group_field_id = col
            break
print(f"Selected group/categorical field: {group_field_id}")

df = dataframes[primary_record_set_id]

# Ensure numeric dtype for selected field
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outliers (only if reasonable)
na_mask = df[numeric_field_id].notna()
threshold = np.percentile(df.loc[na_mask, numeric_field_id], 95)  # keep top 5% for display

filtered_df = df[df[numeric_field_id] < threshold]
print(f"Filtered {numeric_field_id} below 95th percentile ({threshold:.2f}). Remaining rows: {len(filtered_df)}\n")

# Normalize
mu = filtered_df[numeric_field_id].mean()
sigma = filtered_df[numeric_field_id].std()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mu)/sigma

print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the group (categorical) field if present
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std', 'min', 'max'])
    print(f"\nGrouped by {group_field_id}:")
    print(grouped)

## 5. Visualization
Visualize data distribution for the selected numeric field, and the group means by selected category. (Requires `matplotlib`/`seaborn`.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end processing of the FAIR² second primary CRC dataset using `mlcroissant`. We loaded metadata and record sets via their `@id`s, explored and described available fields, transformed the data, and visualized key clinicopathological patterns using pandas and seaborn.

All exploration referenced entities by their schema `@id`, per Croissant and FAIR best practices. For more advanced analytics, consult the Croissant schema documentation and the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).